# 🔋 E-Waste RecoveryAI — Full Training Notebook

| Stage | Model | Task |
|-------|-------|------|
| 1 | EfficientNet-B0 | 12-class device classification |
| 2 | YOLOv8n | PCB component detection |
| 3 | YOLOv8n | RAM/SSD component detection |

> ⚡ **Set runtime to GPU first:** `Runtime → Change runtime type → T4 GPU`

---
## 0️⃣  Setup

In [ ]:
import subprocess, sys
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("❌ No GPU! Go to Runtime → Change runtime type → GPU")
print(result.stdout[:400])

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("✅ Google Drive mounted")

In [ ]:
!pip install -q \
    ultralytics>=8.0.0 \
    imagehash>=4.3.0 \
    scikit-learn>=1.3.0 \
    tqdm>=4.66.0 \
    pyyaml>=6.0 \
    pytorch-grad-cam>=1.5.0 \
    seaborn>=0.12.0 \
    huggingface-hub>=0.22.0
print("✅ Packages installed")

In [ ]:
import os
from pathlib import Path

# GitHub repo URL
GITHUB_REPO = "https://github.com/Prithviisss/AI-Based-E-Waste-Device-and-Component-Detection-with-Recovery-and-Reuse-Recommendation.git"

PROJECT_DIR = Path("/content/ewaste")
if not PROJECT_DIR.exists():
    !git clone {GITHUB_REPO} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f"✅ Working dir: {os.getcwd()}")

In [ ]:
import shutil

DATA_RAW     = PROJECT_DIR / "data" / "raw"
DATA_PROC    = PROJECT_DIR / "data" / "processed"
REPORTS      = PROJECT_DIR / "reports"
MY_DRIVE     = Path("/content/drive/MyDrive")  # Root of Google Drive
DRIVE_MODELS = MY_DRIVE / "ewaste_trained_models"

for p in [DATA_RAW, DATA_PROC, REPORTS/"runs", REPORTS/"figures", DRIVE_MODELS]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Directories ready")

---
## 📥  Datasets

| # | Dataset | How it gets here |
|---|---------|------------------|
| 1 | Kaggle E-Waste Classification | **You upload to Drive → copied here** |
| 2 | HuggingFace PCB Detection | **Auto-downloaded below (no API key)** |
| 3 | Roboflow RAM/SSD | **You upload to Drive → copied here** |

### 📦 Dataset 1 — Kaggle E-Waste Classification (from your Drive)

In [ ]:
import time

# Automatically search for the Kaggle classification folder in MyDrive
ds1_candidates = [
    MY_DRIVE / "archive (2)",
    MY_DRIVE / "archive",
    MY_DRIVE / "dataset1_classification",
    MY_DRIVE / "dataset1",
    MY_DRIVE / "ewaste_project" / "dataset1_classification",
    MY_DRIVE / "e-waste-image-dataset",
]
for item in MY_DRIVE.iterdir():
    if item.is_dir():
        low = item.name.lower()
        if any(k in low for k in ["archive", "dataset1", "kaggle", "classif", "e-waste"]):
            if item not in ds1_candidates:
                ds1_candidates.append(item)

DS1_DRIVE = None
for cand in ds1_candidates:
    if cand.exists() and cand.is_dir():
        DS1_DRIVE = cand
        break

DS1_LOCAL = DATA_RAW / "dataset1_classification"

if DS1_DRIVE is None:
    print("❌ Could not automatically locate Kaggle dataset folder in MyDrive.")
    print("Folders found in MyDrive:")
    for d in sorted([d.name for d in MY_DRIVE.iterdir() if d.is_dir()]):
        print(f"  📁 {d}")
    raise FileNotFoundError("Please set DS1_DRIVE to your folder name in MyDrive.")

print(f"🔍 Found Dataset 1 at: {DS1_DRIVE}")
if DS1_LOCAL.exists() and any(DS1_LOCAL.iterdir()):
    print("✅ Dataset 1 already local — skipping copy")
else:
    print("📦 Copying Dataset 1 from Drive to local...")
    t0 = time.time()
    shutil.copytree(str(DS1_DRIVE), str(DS1_LOCAL), dirs_exist_ok=True)
    n = sum(1 for _ in DS1_LOCAL.rglob('*') if _.is_file())
    elapsed = time.time() - t0
    print(f"✅ {n} files copied in {elapsed:.0f}s")


In [ ]:
# Inspect structure and auto-flatten if Kaggle gave Train/Test/Val subfolders
top_dirs = [d for d in DS1_LOCAL.iterdir() if d.is_dir()]
print(f"Top-level folders: {[d.name for d in top_dirs]}")

split_names = {"train", "test", "val", "validation", "Train", "Test", "Val", "Validation"}
split_dirs  = [d for d in top_dirs if d.name in split_names]

FLAT_DIR = DATA_RAW / "dataset1_flat"

if not split_dirs:
    print("✅ Already flat — class folders are at top level")
    FLAT_DIR = DS1_LOCAL
else:
    print(f"🔄 Found split folders {[d.name for d in split_dirs]} — flattening...")
    FLAT_DIR.mkdir(parents=True, exist_ok=True)
    for split_dir in split_dirs:
        for cls_dir in split_dir.iterdir():
            if not cls_dir.is_dir(): continue
            dst_cls = FLAT_DIR / cls_dir.name
            dst_cls.mkdir(exist_ok=True)
            for img in cls_dir.iterdir():
                if img.is_file():
                    dst = dst_cls / img.name
                    if not dst.exists():
                        shutil.copy2(img, dst)
    print("✅ Flattened. Class counts:")
    for cls in sorted(FLAT_DIR.iterdir()):
        if cls.is_dir():
            print(f"   {cls.name:<20} {sum(1 for _ in cls.glob('*') if _.is_file())} images")

DS1_LOCAL = FLAT_DIR

### 📦 Dataset 2 — PCB Component Detection (Roboflow)


In [ ]:
# 1. Automatically locate the PCB dataset folder in Google Drive
ds2_candidates = [
    MY_DRIVE / "pcb-board-component.v2i.yolov8",
    MY_DRIVE / "pcb-board-component",
    MY_DRIVE / "dataset2_pcb",
    MY_DRIVE / "dataset2",
    MY_DRIVE / "pcb",
    MY_DRIVE / "PCB-Board-Component",
    MY_DRIVE / "PCB-Component-Detection",
    MY_DRIVE / "PCB-Components",
    MY_DRIVE / "ewaste_project" / "dataset2_pcb",
]

for item in MY_DRIVE.iterdir():
    if item.is_dir():
        low = item.name.lower()
        if any(k in low for k in ["pcb", "component", "dataset2"]):
            if "dataset1" not in low and "ram" not in low and "ssd" not in low and item not in ds2_candidates:
                ds2_candidates.append(item)

DS2_DRIVE = None
for cand in ds2_candidates:
    if cand.exists() and cand.is_dir():
        DS2_DRIVE = cand
        break

DS2_LOCAL = DATA_RAW / "dataset2_pcb"

if DS2_DRIVE is None:
    print("❌ Could not automatically locate the PCB dataset folder in Google Drive.")
    print("Folders found in MyDrive:")
    for d in sorted([d.name for d in MY_DRIVE.iterdir() if d.is_dir()]):
        print(f"  📁 {d}")
    raise FileNotFoundError("Please upload your extracted Roboflow PCB dataset folder to Google Drive.")
else:
    print(f"📁 Found Dataset 2 in Drive at: {DS2_DRIVE}")
    if DS2_LOCAL.exists() and any(DS2_LOCAL.iterdir()):
        n = sum(1 for _ in DS2_LOCAL.rglob('*') if _.is_file())
        print(f"✅ Dataset 2 already in local storage ({n:,} files) — skipping copy")
    else:
        print(f"📋 Copying {DS2_DRIVE.name} to {DS2_LOCAL}...")
        t0 = time.time()
        shutil.copytree(str(DS2_DRIVE), str(DS2_LOCAL), dirs_exist_ok=True)
        n = sum(1 for _ in DS2_LOCAL.rglob('*') if _.is_file())
        print(f"✅ Done! {n:,} files copied in {time.time() - t0:.1f}s")


In [ ]:
import yaml

# Auto-detect if user extracted into a nested subfolder (e.g. dataset2_pcb/pcb-board-component/...)
yaml_candidates = list(DS2_LOCAL.rglob("data.yaml")) + list(DS2_LOCAL.rglob("dataset.yaml"))
if yaml_candidates:
    # If data.yaml is inside a nested subfolder, adjust DS2_LOCAL to that root
    yp = yaml_candidates[0]
    if yp.parent != DS2_LOCAL and (yp.parent / 'train').exists():
        print(f"🔄 Pointing DS2_LOCAL to inner dataset directory: {yp.parent.name}")
        DS2_LOCAL = yp.parent
    info = yaml.safe_load(yp.read_text())
    print("📂 Dataset 2 — contents:")
    for item in sorted(DS2_LOCAL.iterdir()):
        n = sum(1 for _ in item.rglob('*') if _.is_file()) if item.is_dir() else 1
        icon = "📁" if item.is_dir() else "📄"
        print(f"   {icon} {item.name}  ({n} files)")
    nc = info.get('nc', len(info.get('names', [])))
    print(f"\n📋 PCB Classes ({nc}): {info.get('names')}")
else:
    print("⚠ No data.yaml found in Dataset 2 yet. Inspecting directory contents:")
    for item in sorted(DS2_LOCAL.iterdir()):
        print(f"   {item.name}")


### 📦 Dataset 3 — Roboflow RAM/SSD (from your Drive)

In [ ]:
# Automatically search for the Roboflow RAM/SSD folder in MyDrive
ds3_candidates = [
    MY_DRIVE / "RAM-SSD-Detector.v2-roboflow-instant-2--eval-.yolov8",
    MY_DRIVE / "dataset3_ramssd",
    MY_DRIVE / "dataset3",
    MY_DRIVE / "RAM-SSD-Detector",
    MY_DRIVE / "ewaste_project" / "dataset3_ramssd",
]
for item in MY_DRIVE.iterdir():
    if item.is_dir() and any(k in item.name.lower() for k in ["dataset3", "ram", "ssd", "roboflow"]):
        if item not in ds3_candidates:
            ds3_candidates.append(item)

DS3_DRIVE = None
for cand in ds3_candidates:
    if cand.exists() and cand.is_dir():
        DS3_DRIVE = cand
        break

DS3_LOCAL = DATA_RAW / "dataset3_ramssd"

if DS3_DRIVE is None:
    print("❌ Could not automatically locate RAM/SSD dataset folder in MyDrive.")
    print("Folders found in MyDrive:")
    for d in sorted([d.name for d in MY_DRIVE.iterdir() if d.is_dir()]):
        print(f"  📁 {d}")
    raise FileNotFoundError("Please set DS3_DRIVE to your folder name in MyDrive.")

print(f"🔍 Found Dataset 3 at: {DS3_DRIVE}")
if DS3_LOCAL.exists() and any(DS3_LOCAL.iterdir()):
    print("✅ Dataset 3 already local — skipping copy")
else:
    print("📦 Copying Dataset 3 from Drive...")
    t0 = time.time()
    shutil.copytree(str(DS3_DRIVE), str(DS3_LOCAL), dirs_exist_ok=True)
    n = sum(1 for _ in DS3_LOCAL.rglob('*') if _.is_file())
    elapsed = time.time() - t0
    print(f"✅ {n} files copied in {elapsed:.0f}s")


In [ ]:
print("📂 Dataset 3 — contents:")
for item in sorted(DS3_LOCAL.iterdir()):
    n = sum(1 for _ in item.rglob("*") if _.is_file()) if item.is_dir() else 1
    icon = "📁" if item.is_dir() else "📄"
    print(f"   {icon} {item.name}  ({n} files)")

for yname in ["data.yaml", "dataset.yaml"]:
    yp = DS3_LOCAL / yname
    if yp.exists():
        info = yaml.safe_load(yp.read_text())
        print(f"\n📋 Classes ({info.get('nc')}): {info.get('names')}")
        break

---
## 1️⃣  Stage 1 — EfficientNet-B0 Device Classifier

In [ ]:
from src.classify.dedupe import dedupe

DEDUPED_DIR = DATA_PROC / "classification" / "deduped"

if DEDUPED_DIR.exists() and any(DEDUPED_DIR.iterdir()):
    print("✅ Deduped already — skipping")
else:
    s = dedupe(input_dir=DS1_LOCAL, output_dir=DEDUPED_DIR, threshold=10)
    print(f"\n📊 {s['total_original']:,} → {s['total_kept']:,} kept  ({s['total_removed']:,} removed)")

In [ ]:
from src.classify.split import split_dataset

SPLIT_DIR = DATA_PROC / "classification" / "split"

if SPLIT_DIR.exists() and any(SPLIT_DIR.iterdir()):
    print("✅ Split already — skipping")
else:
    split_dataset(input_dir=DEDUPED_DIR, output_dir=SPLIT_DIR)

for s in ["train", "val", "test"]:
    n = sum(1 for _ in (SPLIT_DIR/s).rglob("*") if _.is_file())
    print(f"  {s}: {n:,}")

In [ ]:
from src.classify.train import train as train_classifier

CLASSIFY_OUT = REPORTS / "runs" / "classify"
CLASSIFY_CKPT = CLASSIFY_OUT / "best_classifier.pt"
DRIVE_CKPT = DRIVE_MODELS / "best_classifier.pt"

EPOCHS     = 30    # ← set to 1 for a quick smoke test
BATCH_SIZE = 32
LR         = 1e-3

# Check if already trained previously and saved to Drive or local
if CLASSIFY_CKPT.exists():
    print(f"✅ Stage 1 classifier already exists locally ({CLASSIFY_CKPT}) — skipping re-training!")
elif DRIVE_CKPT.exists():
    print(f"✅ Found trained classifier in Google Drive! Copying from {DRIVE_CKPT}...")
    CLASSIFY_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_CKPT, CLASSIFY_CKPT)
    print(f"✅ Model loaded successfully into {CLASSIFY_CKPT} — skipping re-training!")
else:
    print(f"🚀 Training EfficientNet-B0 — {EPOCHS} epochs")
    model, class_names = train_classifier(
        data_dir=SPLIT_DIR, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, output_dir=CLASSIFY_OUT
    )
    print(f"\n✅ Done → {CLASSIFY_CKPT}")


In [ ]:
from src.classify.eval import evaluate as eval_classifier

acc, macro_f1, _ = eval_classifier(data_dir=SPLIT_DIR, ckpt=CLASSIFY_CKPT, batch_size=BATCH_SIZE)
print(f"\n{'='*50}")
print(f"📊 STAGE 1  |  Accuracy={acc:.4f} ({acc*100:.1f}%)  |  Macro F1={macro_f1:.4f}")
print(f"{'='*50}")

In [ ]:
import json, matplotlib.pyplot as plt

h = json.loads((CLASSIFY_OUT / "history.json").read_text())
ep = [x["epoch"] for x in h]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EfficientNet-B0 Training", fontsize=13, fontweight="bold")
ax1.plot(ep, [x["train_loss"] for x in h], label="Train", color="#4F86C6")
ax1.plot(ep, [x["val_loss"]   for x in h], label="Val",   color="#E05C5C", ls="--")
ax1.set_title("Loss"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, [x["train_acc"] for x in h], label="Train", color="#4F86C6")
ax2.plot(ep, [x["val_acc"]   for x in h], label="Val",   color="#E05C5C", ls="--")
ax2.set_title("Accuracy"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS/"figures"/"s1_curves.png", dpi=150)
plt.show()

In [ ]:
shutil.copy2(CLASSIFY_CKPT, DRIVE_MODELS / "best_classifier.pt")
print(f"✅ Saved to Drive → {DRIVE_MODELS / 'best_classifier.pt'}")

---
## 2️⃣  Stage 2 — YOLOv8 PCB Component Detector

In [ ]:
from src.detect_pcb.prepare import prepare as prepare_pcb

PCB_PROC = DATA_PROC / "pcb_detect"
PCB_YAML = PCB_PROC / "dataset.yaml"

if PCB_YAML.exists():
    print("✅ PCB dataset.yaml exists — skipping prepare")
else:
    prepare_pcb(raw_dir=DS2_LOCAL, out_dir=PCB_PROC)

info = yaml.safe_load(PCB_YAML.read_text())
print(f"📋 PCB — {info['nc']} classes: {info['names']}")

In [ ]:
from src.detect_pcb.train import train as train_pcb

PCB_OUT    = REPORTS / "runs" / "detect_pcb"
PCB_EPOCHS = 100      # ← set to 1 for smoke test
PCB_BATCH  = 16
PCB_MODEL  = "yolov8n.pt"

print(f"🚀 Training YOLOv8 PCB Detector — {PCB_EPOCHS} epochs")
pcb_best = train_pcb(
    dataset_yaml=PCB_YAML, model_name=PCB_MODEL,
    epochs=PCB_EPOCHS, batch=PCB_BATCH, output_dir=PCB_OUT
)
print(f"\n✅ Done → {pcb_best}")

In [ ]:
from src.detect_pcb.eval import evaluate as eval_pcb

pcb_m = eval_pcb(dataset_yaml=PCB_YAML, weights=pcb_best)
print(f"\n{'='*50}")
print(f"📊 STAGE 2  |  mAP@50={pcb_m.get('map50','N/A')}  |  mAP@50-95={pcb_m.get('map','N/A')}")
print(f"{'='*50}")

In [ ]:
shutil.copy2(pcb_best, DRIVE_MODELS / "best_pcb_detector.pt")
print(f"✅ Saved to Drive → {DRIVE_MODELS / 'best_pcb_detector.pt'}")

---
## 3️⃣  Stage 3 — YOLOv8 RAM/SSD Component Detector

In [ ]:
from src.detect_ramssd.prepare import prepare as prepare_ramssd

RAMSSD_PROC = DATA_PROC / "ramssd_detect"
RAMSSD_YAML = RAMSSD_PROC / "dataset.yaml"
USE_FALLBACK = False

if RAMSSD_YAML.exists():
    print("✅ RAM/SSD dataset.yaml exists — skipping prepare")
else:
    prepare_ramssd(raw_dir=DS3_LOCAL, out_dir=RAMSSD_PROC, fallback=USE_FALLBACK)

info = yaml.safe_load(RAMSSD_YAML.read_text())
print(f"📋 RAM/SSD — {info['nc']} classes: {info['names']}")

In [ ]:
from src.detect_ramssd.train import train as train_ramssd

RAMSSD_OUT    = REPORTS / "runs" / "detect_ramssd"
RAMSSD_EPOCHS = 100     # ← set to 1 for smoke test
RAMSSD_BATCH  = 8
RAMSSD_MODEL  = "yolov8n.pt"

print(f"🚀 Training YOLOv8 RAM/SSD Detector — {RAMSSD_EPOCHS} epochs")
ramssd_best = train_ramssd(
    dataset_yaml=RAMSSD_YAML, model_name=RAMSSD_MODEL,
    epochs=RAMSSD_EPOCHS, batch=RAMSSD_BATCH, output_dir=RAMSSD_OUT
)
print(f"\n✅ Done → {ramssd_best}")

In [ ]:
from src.detect_ramssd.eval import evaluate as eval_ramssd

ramssd_m = eval_ramssd(dataset_yaml=RAMSSD_YAML, weights=ramssd_best)
print(f"\n{'='*50}")
print(f"📊 STAGE 3  |  mAP@50={ramssd_m.get('map50','N/A')}  |  mAP@50-95={ramssd_m.get('map','N/A')}")
print(f"{'='*50}")

In [ ]:
shutil.copy2(ramssd_best, DRIVE_MODELS / "best_ramssd_detector.pt")
print(f"✅ Saved to Drive → {DRIVE_MODELS / 'best_ramssd_detector.pt'}")

---
## 4️⃣  Final Summary & Download Weights

In [ ]:
import json
print("="*60)
print("🎉  E-WASTE RECOVERYAI — TRAINING COMPLETE")
print("="*60)
mf = REPORTS / "metrics.json"
if mf.exists():
    m = json.loads(mf.read_text())
    if "classifier" in m:
        c = m["classifier"]
        print(f"\n📦 Stage 1  Acc={c.get('accuracy','?')}  F1={c.get('macro_f1','?')}")
    if "detect_pcb" in m:
        print(f"🔧 Stage 2  mAP@50={m['detect_pcb'].get('map50','?')}")
    if "detect_ramssd" in m:
        print(f"💾 Stage 3  mAP@50={m['detect_ramssd'].get('map50','?')}")
print(f"\n📁 Weights at: {DRIVE_MODELS}")

In [ ]:
from google.colab import files

for w in [
    CLASSIFY_OUT / "best_classifier.pt",
    PCB_OUT / "train" / "weights" / "best.pt",
    RAMSSD_OUT / "train" / "weights" / "best.pt",
    REPORTS / "metrics.json",
]:
    if w.exists():
        print(f"⬇️  {w.name}")
        files.download(str(w))
    else:
        print(f"⚠️  Not found: {w}")